In [1]:
import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('../dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

../dataset/customers.csv
../dataset/payments.csv
../dataset/baseline.ipynb
../dataset/products.csv
../dataset/web_traffic.csv
../dataset/promotions.csv
../dataset/order_items.csv
../dataset/geography.csv
../dataset/shipments.csv
../dataset/sales.csv
../dataset/returns.csv
../dataset/inventory.csv
../dataset/sample_submission.csv
../dataset/reviews.csv
../dataset/orders.csv


**Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)**

In [2]:
import pandas as pd 

orders = pd.read_csv("../dataset/orders.csv")
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders_sorted = orders.sort_values(['customer_id', 'order_date'])
multi_order = orders_sorted.groupby('customer_id').filter(lambda x: len(x) > 1)
gaps = multi_order.groupby('customer_id')['order_date'].diff().dt.days.dropna()
print(gaps.median())

144.0


**Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price−cogs)/price?**

In [3]:
products = pd.read_csv("../dataset/products.csv")
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
print(products.groupby('segment')['gross_margin'].mean().idxmax())

Standard


**Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?**

In [4]:
returns = pd.read_csv('../dataset/returns.csv')
products = pd.read_csv('../dataset/products.csv')

merged = returns.merge(products[['product_id', 'category']], on='product_id')
streetwear = merged[merged['category'] == 'Streetwear']
print(streetwear['return_reason'].value_counts().idxmax())

wrong_size


**Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung 
bình(bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?**

In [5]:
web = pd.read_csv('../dataset/web_traffic.csv')
print(web.groupby('traffic_source')['bounce_rate'].mean().idxmin())

email_campaign


**Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?**

In [6]:
order_items = pd.read_csv('../dataset/order_items.csv', low_memory=False)
pct = order_items['promo_id'].notna().mean() * 100
print(f"{pct:.1f}%")

38.7%


**Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)**

In [7]:
customers = pd.read_csv('../dataset/customers.csv')
orders = pd.read_csv('../dataset/orders.csv')
cust = customers[customers['age_group'].notna()]
order_counts = orders.groupby('customer_id')['order_id'].count().reset_index()
order_counts.columns = ['customer_id', 'order_count']
merged = cust.merge(order_counts, on='customer_id', how='left').fillna(0)
print(merged.groupby('age_group')['order_count'].mean().idxmax())

55+


**Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?**

In [8]:
geo = pd.read_csv('../dataset/geography.csv')

# Tính revenue từng dòng
order_items['revenue'] = order_items['unit_price'] * order_items['quantity'] - order_items['discount_amount']

# Join: order_items -> orders -> geography
merged = order_items.merge(orders[['order_id', 'zip']], on='order_id')
merged = merged.merge(geo[['zip', 'region']], on='zip')

print(merged.groupby('region')['revenue'].sum().idxmax())
print(merged.groupby('region')['revenue'].sum())

East
region
Central    4.719491e+09
East       7.291151e+09
West       3.670227e+09
Name: revenue, dtype: float64


**Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?**

In [9]:
orders = pd.read_csv('../dataset/orders.csv')
payments = pd.read_csv('../dataset/payments.csv')

cancelled = orders[orders['order_status'] == 'cancelled'][['order_id']]
merged = cancelled.merge(payments, on='order_id')
print(merged['payment_method'].value_counts().idxmax())
print(merged['payment_method'].value_counts())

credit_card
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


**Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?**

In [10]:
returns = pd.read_csv('../dataset/returns.csv')
products = pd.read_csv('../dataset/products.csv')
order_items = pd.read_csv('../dataset/order_items.csv', low_memory=False)


items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id')
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')

sizes = ['S', 'M', 'L', 'XL']
total = items_with_size[items_with_size['size'].isin(sizes)].groupby('size').size()
returned = returns_with_size[returns_with_size['size'].isin(sizes)].groupby('size').size()

rate = returned / total
print(rate.idxmax())
print(rate)

S
size
L     0.056250
M     0.055660
S     0.056515
XL    0.055200
dtype: float64


**Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?**

In [11]:
payments = pd.read_csv('../dataset/payments.csv')

print(payments.groupby('installments')['payment_value'].mean().idxmax())
print(payments.groupby('installments')['payment_value'].mean())

6
installments
1     24113.274166
2       708.473729
3     24399.635486
6     24446.654403
12    24245.772694
Name: payment_value, dtype: float64
